#Implementing GEORGE Method to Remedy Spurious Correlations in SpuCoMNIST

In [5]:
!pip install spuco


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 84.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 7.7 MB/s eta 0:00:00
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44284 sha256=9642f98bbb90e63efc193bbdfb99ca503c37123df35d0ab68125560c34224fab
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [7]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# Standard Training Using ERM2

> Add blockquote



In [8]:
from spuco.datasets import SpuCoMNIST, SpuriousFeatureDifficulty
import torchvision.transforms as T

difficulty = SpuriousFeatureDifficulty.MAGNITUDE_LARGE

classes = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]

trainset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    spurious_correlation_strength=0.99,
    classes=classes,
    split="train"
)
trainset.initialize()

valset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="val"
)
valset.initialize()


testset = SpuCoMNIST(
    root="/data/mnist/",
    spurious_feature_difficulty=difficulty,
    classes=classes,
    split="test"
)
testset.initialize()


100%|██████████| 9.91M/9.91M [00:00<00:00, 16.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 491kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.4MB/s]


In [9]:
print(len(valset))
print(len(trainset))
print(len(testset))

11996
48004
10000


Creating Data Loaders

In [10]:
from torch.utils.data import DataLoader

train_dl = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)
valid_dl = DataLoader(valset, batch_size=64, num_workers=2)

Initializing Model and Training Loop

```
# This is formatted as code
```



In [14]:
from spuco.models import model_factory
from spuco.robust_train import ERM
from torch.optim import SGD
import torch.nn as nn



#model_factory(arch: str, input_shape: Tuple[int, int, int], num_classes: int, pretrained: bool = True)
model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device) #using pretrained lenet
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
num_epochs=3
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

for epoch in range(num_epochs):
    model.train() #training mode
    running_loss=0
    for xb, yb in train_dl: #goes through the images and masks
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad() #zero the gradients each batch
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_dl)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.3f}")



Epoch 1/3, Loss: 0.244
Epoch 2/3, Loss: 0.068
Epoch 3/3, Loss: 0.060


Testing

In [15]:
from spuco.evaluate import Evaluator

#using spuco function
evaluator = Evaluator(
    testset=testset,
    group_partition=testset.group_partition,
    group_weights=trainset.group_weights,
    batch_size=64,
    model=model,
    device=device,
    verbose=True
)
evaluator.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Evaluating group-wise accuracy:   4%|▍         | 1/25 [00:00<00:08,  2.95it/s]

Group (0, 0) Accuracy: 100.0


Evaluating group-wise accuracy:   8%|▊         | 2/25 [00:00<00:06,  3.44it/s]

Group (0, 1) Accuracy: 0.0


Evaluating group-wise accuracy:  12%|█▏        | 3/25 [00:00<00:05,  3.73it/s]

...

Evaluating group-wise accuracy:  96%|█████████▌| 24/25 [00:07<00:00,  3.60it/s]

Group (4, 3) Accuracy: 0.0


Evaluating group-wise accuracy: 100%|██████████| 25/25 [00:07<00:00,  3.40it/s]

Group (4, 4) Accuracy: 99.74747474747475


{(0, 0): 100.0,
 (0, 1): 0.0,
 (0, 2): 0.0,
 (0, 3): 6.382978723404255,
 (0, 4): 21.513002364066192,
 (1, 0): 0.0,
 (1, 1): 100.0,
 (1, 2): 0.0,
 (1, 3): 0.0,
 (1, 4): 0.0,
 (2, 0): 0.0,
 (2, 1): 0.0,
 (2, 2): 100.0,
 (2, 3): 0.26666666666666666,
 (2, 4): 0.0,
 (3, 0): 0.0,
 (3, 1): 0.0,
 (3, 2): 0.0,
 (3, 3): 100.0,
 (3, 4): 0.0,
 (4, 0): 9.319899244332493,
 (4, 1): 0.0,
 (4, 2): 0.0,
 (4, 3): 0.0,
 (4, 4): 99.74747474747475}

Saving Model Weights

In [16]:
from google.colab import drive
drive.mount('/content/drive')

torch.save(model.state_dict(), "/content/drive/MyDrive/erm2_weights.pth")

Mounted at /content/drive


In [17]:
#locally
torch.save(model.state_dict(), "erm2_weights.pth")


#Cluster inputs based on the output they produce

In [18]:
from spuco.group_inference import Cluster, ClusterAlg
#first collecting the outputs
model.eval()
all_outputs = []
with torch.no_grad():
    for xb, _ in train_dl:
        xb = xb.to(device)
        preds = model(xb)
        all_outputs.append(preds.cpu())
logits = torch.cat(all_outputs, dim=0)
cluster = Cluster(
    Z=logits,
    class_labels=trainset.labels,
    cluster_alg=ClusterAlg.KMEANS, #k means method for clustering
    num_clusters=2,
    device=device,
    verbose=True
)
group_partition = cluster.infer_groups()

Clustering class-wise: 100%|██████████| 5/5 [00:00<00:00, 17.86it/s]


In [19]:
for key in sorted(group_partition.keys()):
    print(key, len(group_partition[key]))

(0, 0) 4024
(0, 1) 6109
(1, 0) 3831
(1, 1) 5841
(2, 0) 3568
(2, 1) 5443
(3, 0) 5829
(3, 1) 3918
(4, 0) 5641
(4, 1) 3800


#Retraining using "Group-Balancing"

> Add blockquote



In [ ]:
#classGroupBalanceBatchERM(model: Module, trainset: Dataset, group_partition: Dict, batch_size: int, optimizer: Optimizer, num_epochs: int, device: device = device(type='cpu'), val_evaluator: Evaluator | None = None, verbose=False, use_wandb=False

In [20]:
from spuco.robust_train import GroupBalanceBatchERM
model = model_factory("lenet", trainset[0][0].shape, trainset.num_classes).to(device) #using pretrained lenet

group_balance = GroupBalanceBatchERM(
    model=model,
    num_epochs=3,
    trainset=trainset,
    group_partition=group_partition,
    batch_size=64,
    optimizer=SGD(model.parameters(), lr=0.01,  momentum=0.9),
    device=device,
    verbose=True
)
group_balance.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
Epoch 0:   2%|▏         | 14/751 [00:00<00:11, 61.63batch/s, accuracy=60.9375%, loss=1.54]

GB | Epoch 0 | Loss: 1.606658935546875 | Accuracy: 18.75%
GB | Epoch 0 | Loss: 1.6116249561309814 | Accuracy: 10.9375%
GB | Epoch 0 | Loss: 1.6154601573944092 | Accuracy: 18.75%
GB | Epoch 0 | Loss: 1.61902916431427 | Accuracy: 9.375%
GB | Epoch 0 | Loss: 1.6095154285430908 | Accuracy: 25.0%
GB | Epoch 0 | Loss: 1.6077128648757935 | Accuracy: 21.875%
GB | Epoch 0 | Loss: 1.6067211627960205 | Accuracy: 21.875%
GB | Epoch 0 | Loss: 1.6044857501983643 | Accuracy: 21.875%
GB | Epoch 0 | Loss: 1.6026413440704346 | Accuracy: 23.4375%
GB | Epoch 0 | Loss: 1.601063847541809 | Accuracy: 34.375%
GB | Epoch 0 | Loss: 1.5995440483093262 | Accuracy: 45.3125%
GB | Epoch 0 | Loss: 1.6014677286148071 | Accuracy: 39.0625%
GB | Epoch 0 | Loss: 1.6020503044128418 | Accuracy: 32.8125%
GB | Epoch 0 | Loss: 1.6076545715332031 | Accuracy: 40.625%
GB | Epoch 0 | Loss: 1.5913769006729126 | Accuracy: 54.6875%
GB | Epoch 0 | Loss: 1.607438564300537 | Accuracy: 31.25%
GB | Epoch 0 | Loss: 1.6052147150039673 | Acc

Epoch 0:   6%|▌         | 45/751 [00:00<00:06, 114.13batch/s, accuracy=85.9375%, loss=0.981]

GB | Epoch 0 | Loss: 1.5441837310791016 | Accuracy: 48.4375%
GB | Epoch 0 | Loss: 1.5262724161148071 | Accuracy: 45.3125%
GB | Epoch 0 | Loss: 1.5216312408447266 | Accuracy: 35.9375%
GB | Epoch 0 | Loss: 1.5086331367492676 | Accuracy: 46.875%
GB | Epoch 0 | Loss: 1.4931045770645142 | Accuracy: 40.625%
GB | Epoch 0 | Loss: 1.4937543869018555 | Accuracy: 45.3125%
GB | Epoch 0 | Loss: 1.4646270275115967 | Accuracy: 51.5625%
GB | Epoch 0 | Loss: 1.409745216369629 | Accuracy: 50.0%
GB | Epoch 0 | Loss: 1.4089813232421875 | Accuracy: 40.625%
GB | Epoch 0 | Loss: 1.3500503301620483 | Accuracy: 43.75%
GB | Epoch 0 | Loss: 1.2974529266357422 | Accuracy: 48.4375%
GB | Epoch 0 | Loss: 1.261980414390564 | Accuracy: 39.0625%
GB | Epoch 0 | Loss: 1.1941767930984497 | Accuracy: 43.75%
GB | Epoch 0 | Loss: 1.126163125038147 | Accuracy: 43.75%
GB | Epoch 0 | Loss: 1.0448721647262573 | Accuracy: 45.3125%
GB | Epoch 0 | Loss: 1.0670740604400635 | Accuracy: 35.9375%
GB | Epoch 0 | Loss: 1.1379473209381104

Epoch 0:  11%|█         | 80/751 [00:00<00:04, 144.92batch/s, accuracy=98.4375%, loss=0.199]

...

GB | Epoch 2 | Loss: 0.16461411118507385 | Accuracy: 96.875%
GB | Epoch 2 | Loss: 0.19614604115486145 | Accuracy: 96.875%
GB | Epoch 2 | Loss: 0.09201046824455261 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.17463767528533936 | Accuracy: 96.875%
GB | Epoch 2 | Loss: 0.01203937642276287 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.177289217710495 | Accuracy: 96.875%
GB | Epoch 2 | Loss: 0.012731011025607586 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.01581256464123726 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.013164560310542583 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.010485381819307804 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.016462113708257675 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.22702711820602417 | Accuracy: 95.3125%
GB | Epoch 2 | Loss: 0.1035982221364975 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.09899908304214478 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.0157703198492527 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.010575633496046066 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.0126

Epoch 2:  97%|█████████▋| 729/751 [00:06<00:00, 129.04batch/s, accuracy=100.0%, loss=0.0172]

GB | Epoch 2 | Loss: 0.1171216294169426 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.09915825724601746 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.10887356847524643 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.01202329620718956 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.17814521491527557 | Accuracy: 96.875%
GB | Epoch 2 | Loss: 0.009347017854452133 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.010395422577857971 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.010664640925824642 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.08732374012470245 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.09499093145132065 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.011592278257012367 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.09398382902145386 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.14916911721229553 | Accuracy: 96.875%
GB | Epoch 2 | Loss: 0.012698153965175152 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.11150208860635757 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.013672593981027603 | Accuracy: 100.0%
GB | Epoch 2 | Loss

Epoch 2: 100%|██████████| 751/751 [00:06<00:00, 116.83batch/s, accuracy=100.0%, loss=0.00462]

GB | Epoch 2 | Loss: 0.011736473068594933 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.013483601622283459 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.008862195536494255 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.01275487057864666 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.01191947516053915 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.010539732873439789 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.008774147368967533 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.12597832083702087 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.0075507936999201775 | Accuracy: 100.0%
GB | Epoch 2 | Loss: 0.09924271702766418 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.10901723802089691 | Accuracy: 98.4375%
GB | Epoch 2 | Loss: 0.004623404238373041 | Accuracy: 100.0%
